# Message queues

A queue decouples accepting a job from processing it.

In [ ]:
import asyncio

# A producer puts work in the queue; a worker receives and acknowledges it.
queue = asyncio.Queue()
await queue.put({"id": "job-1", "prompt": "Hello"})

job = await queue.get()
print("Received:", job)
queue.task_done()

## Polished version

A standalone queue exposes the production lifecycle: enqueue, inspect status, reserve a job, and store its result.

In [ ]:
import asyncio
from dataclasses import dataclass
from typing import Optional, Protocol
from uuid import uuid4


# Keep the immutable job payload separate from its changing status.
@dataclass(frozen=True)
class Job:
    id: str
    prompt: str


@dataclass(frozen=True)
class JobStatus:
    id: str
    state: str
    result: Optional[str] = None


# The API producer and background worker share this queue contract.
class JobQueue(Protocol):
    async def enqueue(self, prompt: str) -> str: ...
    async def status(self, job_id: str) -> JobStatus: ...
    async def reserve(self) -> Job: ...
    async def complete(self, job_id: str, result: str) -> None: ...


class MemoryJobQueue:
    def __init__(self) -> None:
        self.pending: asyncio.Queue[Job] = asyncio.Queue()
        self.states: dict[str, JobStatus] = {}

    async def enqueue(self, prompt: str) -> str:
        # Store status before publishing so clients can query immediately.
        job = Job(id=str(uuid4()), prompt=prompt)
        self.states[job.id] = JobStatus(job.id, "queued")
        await self.pending.put(job)
        return job.id

    async def status(self, job_id: str) -> JobStatus:
        return self.states[job_id]

    async def reserve(self) -> Job:
        # Only one worker receives each item removed from this queue.
        job = await self.pending.get()
        self.states[job.id] = JobStatus(job.id, "in_progress")
        return job

    async def complete(self, job_id: str, result: str) -> None:
        # Save the result, then acknowledge that queue processing finished.
        self.states[job_id] = JobStatus(job_id, "complete", result)
        self.pending.task_done()


queue: JobQueue = MemoryJobQueue()
job_id = await queue.enqueue("Explain queues")
print(await queue.status(job_id))

job = await queue.reserve()
await queue.complete(job.id, job.prompt.upper())
print(await queue.status(job_id))

## Applied in this repository

The production [JobQueue](../00P2-project-llm-api/app/queue.py) exposes enqueue and status to FastAPI while ARQ and Redis handle reservation, durability, and result storage.